# 1. First LLM Call and Different Ways of Calling

Learn `invoke`, message-based invocation, prompt-chain invocation, `batch`, `stream`, `ainvoke` and response metadata using real calls.

## 1. Workflow

Input is sent through `ChatOpenAI`; the result is an `AIMessage`. Different calling methods suit single requests, concurrent work, live interfaces and reusable chains.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
llm=ChatOpenAI(model=MODEL_NAME,temperature=0,max_retries=2,timeout=30)
response=llm.invoke("Explain generative AI in two sentences.")
print(response.content)

### 3. Call the model and inspect the response

This cell calls the configured model with the prepared prompt and extracts the returned content.

**Expected result:** A live model response is printed; wording can vary unless deterministic settings are used. Read the output before continuing to the next cell.

In [ ]:
messages=[("system","You are a concise Python trainer."),("human","What is a list comprehension?")]
print(llm.invoke(messages).content)

### 4. Create a reusable prompt template

This cell creates a reusable prompt template with variables so the same workflow can handle different inputs consistently.

**Expected result:** The rendered prompt is sent to the model and the resulting text is displayed. Read the output before continuing to the next cell.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt=ChatPromptTemplate.from_template("Explain {topic} to a {audience} using one example.")
chain=prompt|llm|StrOutputParser()
print(chain.invoke({"topic":"RAG","audience":"beginner"}))

### 5. Process multiple prompts as a batch

This cell submits multiple independent prompts as a batch with controlled concurrency.

**Expected result:** One response is printed for every input item while preserving result order. Read the output before continuing to the next cell.

In [ ]:
questions=["Define token.","Define embedding.","Define context window."]
for answer in llm.batch(questions,config={"max_concurrency":3}):
    print(answer.content)

### 6. Stream the model response

This cell streams the response incrementally, which improves perceived responsiveness in chat interfaces.

**Expected result:** Text appears progressively instead of waiting for the complete response. Read the output before continuing to the next cell.

In [ ]:
for chunk in llm.stream("Give five short benefits of prompt templates."):
    if chunk.content: print(chunk.content,end="",flush=True)

### 7. Run the asynchronous model call

This cell uses the asynchronous LangChain interface so independent requests do not block one another.

**Expected result:** Responses are returned after the awaited operation completes. Read the output before continuing to the next cell.

In [ ]:
# Jupyter supports top-level await.
async_response=await llm.ainvoke("Give one use case of asynchronous LLM calls.")
print(async_response.content)

### 8. Prepare the next processing step

This cell prepares the variables, functions, or validation logic required by the next stage of the example.

**Expected result:** The cell defines reusable objects or prints a small verification result. Read the output before continuing to the next cell.

In [ ]:
print("Usage:",response.usage_metadata)
print("Model metadata:",response.response_metadata)

## Validation and exercises

Confirm every method returns the requested content. Compare `invoke`, `batch`, `stream` and `ainvoke`; change the prompt and observe usage metadata.